In [ ]:
"""
Dynamic Traffic Routing Simulator + Nash Analysis (FULL CODE)

What you get in this one file:
1) Single source-of-truth simulator (same randomness for animation + NE analysis) using CRN.
2) Method A: Empirical ε–NE over STRATEGY TYPES (Aggressive/Strategic/Risk-Averse/Social).
3) Method B: Instantaneous road-choice local NE gap (weighted + decision mass + optional M>=k filter).
4) Better visuals + data for interpretation:
   - Baseline strategy outcome plots (utility, arrivals, congestion waves, regret)
   - Weighted local-NE gap over time + decision mass
   - Road-choice shares over time (stacked)
   - Congestion heatmap (node x time) based on traversal counts
   - Top deviation table (for ε–NE explanation)

Usage:
- Run this file. It will:
  a) run baseline simulation + plots
  b) run Method A epsilon-NE (fast mode by default)
  c) run Method B local NE gap plots
  d) print report-ready summaries

Notes:
- CRN = Common Random Numbers. Randomness is deterministic per (seed, agent_id, node, road, tag)
  so baseline vs deviation comparisons are fair and not RNG-order artifacts.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import seaborn as sns

# =========================
# CONFIG
# =========================
STRATS = ["Aggressive", "Strategic", "Risk-Averse", "Social"]

DEFAULT_T = 500
NODES = 5

BASELINE_STRATEGIES = ["Aggressive"] * 25 + ["Strategic"] * 25 + ["Risk-Averse"] * 25 + ["Social"] * 25
POP_SIZE = len(BASELINE_STRATEGIES)

# =========================
# CRN HELPERS (deterministic randomness)
# =========================
import hashlib

def stable_hash(seed: int, *keys) -> int:
    s = str(seed) + "||" + "||".join(map(str, keys))
    digest = hashlib.blake2b(s.encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, byteorder="little") & 0xFFFFFFFF

def stable_rng(seed: int, *keys):
    return np.random.default_rng(stable_hash(seed, *keys))


def crn_patience(seed: int, agent_id: int, mu=20, sigma=8):
    rng = stable_rng(seed, agent_id, "patience")
    return max(0, int(round(rng.normal(mu, sigma))))

def crn_incident(seed: int, agent_id: int, node: int, road_name: str):
    rng = stable_rng(seed, agent_id, node, road_name, "incident")
    return rng.random()

def crn_yjitter(seed: int, agent_id: int, node: int, road_name: str):
    rng = stable_rng(seed, agent_id, node, road_name, "yjitter")
    return rng.uniform(-0.08, 0.08)


# =========================
# 1) ENVIRONMENT: DYNAMIC ROAD NETWORK
# =========================
class Road:
    def __init__(self, name, node_id, y_level, a_e, b_e, capacity=20, open_at=0, incident_prob=0.05):
        self.name = name
        self.node_id = node_id
        self.y_level = y_level
        self.a_e = a_e
        self.b_e = b_e
        self.capacity = capacity
        self.open_at = open_at
        self.incident_prob = incident_prob
        self.flow = 0

    def travel_time(self):
        congestion = self.b_e * (self.flow if self.flow < self.capacity else self.flow ** 1.8)
        return self.a_e + congestion

    def get_metrics(self, is_social=False):
        t = self.travel_time()
        fuel_cost = 5 + (t * 0.15)
        if is_social:
            return t + (self.flow * self.b_e), fuel_cost
        return t, fuel_cost


def make_network():
    nodes = NODES
    network = {
        i: [
            Road("Slow", i, 0.0, 45, 0.2, capacity=40, incident_prob=0.02),
            Road("Std",  i, 0.5, 25, 0.8, capacity=25, incident_prob=0.15),
            Road("Exp",  i, 1.0, 8,  4.0, capacity=12, open_at=10 + (i * 12), incident_prob=0.10),
        ]
        for i in range(nodes)
    }
    return nodes, network


# =========================
# 2) AGENTS
# =========================
class Agent:
    def __init__(self, agent_id, strategy, patience_limit):
        self.id = agent_id
        self.strategy = strategy
        self.patience_limit = max(0, int(patience_limit))

        self.current_node = 0
        self.state = "WAITING"

        # visualization state
        self.pos_x = 0.0
        self.pos_y = 0.5

        # traversal state
        self.current_road = None
        self.edge_length = 1.0
        self.edge_progress = 0.0
        self.sampled_travel_time = 0.0

        # logs
        self.wait_time = 0
        self.drive_time = 0
        self.fuel_spent = 0
        self.incidents_hit = 0
        self.arrival_time = None
        self.regret = 0.0

    def decide(self, t, roads, seed, decision_events=None):
        avail = [r for r in roads if r.open_at <= t]
        if not avail:
            self.wait_time += 1
            return

        all_costs = [r.get_metrics()[0] for r in avail]
        min_cost = float(min(all_costs))

        if self.wait_time >= self.patience_limit or self.strategy == "Aggressive":
            choice = avail[int(np.argmin(all_costs))]

        elif self.strategy == "Strategic":
            future = sorted([r for r in roads if r.open_at > t], key=lambda r: r.open_at)
            if future:
                best_future = future[0]
                wait_until_open = best_future.open_at - t
                future_best_cost = wait_until_open + best_future.a_e  # optimistic
                if min_cost > future_best_cost:
                    self.wait_time += 1
                    return
            choice = avail[int(np.argmin(all_costs))]

        elif self.strategy == "Social":
            choice = min(avail, key=lambda r: r.get_metrics(is_social=True)[0])

        else:  # Risk-Averse
            safe = [r for r in avail if r.incident_prob <= 0.1]
            choice = min(safe, key=lambda r: r.get_metrics()[0]) if safe else avail[int(np.argmin(all_costs))]

        self.regret = float(choice.get_metrics()[0] - min_cost)

        # record decision batch info BEFORE embarking (for Method B)
        if decision_events is not None:
            base_flows = {r.name: r.flow for r in avail}
            decision_events.append({
                "t": int(t),
                "agent": int(self.id),
                "strategy": self.strategy,
                "node": int(self.current_node),
                "choice": choice.name,
                "regret": float(self.regret),
                "avail": tuple([r.name for r in avail]),
                "base_flows": base_flows,
            })

        self.embark(choice, seed=seed)

    def embark(self, road: Road, seed: int):
        self.state = "TRAVERSING"
        self.current_road = road
        self.edge_progress = 0.0

        t_rem, fuel = road.get_metrics()

        # CRN incident draw
        if crn_incident(seed, self.id, self.current_node, road.name) < road.incident_prob:
            t_rem *= 1.6
            self.incidents_hit += 1

        self.sampled_travel_time = max(1.0, float(t_rem))
        self.fuel_spent += float(fuel)

        # CRN y-jitter
        self.pos_y = road.y_level + crn_yjitter(seed, self.id, self.current_node, road.name)

        road.flow += 1

    def step_traverse(self):
        speed = self.edge_length / self.sampled_travel_time
        self.edge_progress += speed
        self.drive_time += 1
        self.pos_x = self.current_node + min(1.0, self.edge_progress)

        if self.edge_progress >= 1.0:
            # finished this edge
            self.current_road.flow = max(0, self.current_road.flow - 1)
            self.current_road = None
            self.edge_progress = 0.0
            self.sampled_travel_time = 0.0

            self.current_node += 1
            self.pos_x = float(self.current_node)
            return True

        return False


# =========================
# 3) SINGLE SOURCE-OF-TRUTH SIMULATOR
# =========================
def simulate(strategies, seed=0, T=DEFAULT_T, record_full_log=True, record_decisions=True):
    """
    Returns:
      res_df: per-agent final metrics (Utility, Wait, Drive, Fuel, etc.)
      log_df: per-timestep logs (optional; can be large)
      dec_df: per-decision events (for Method B)
      traverse_df: traversal counts per (t,node,road_name) for better visuals
    """
    nodes, network = make_network()
    N = len(strategies)

    agents = []
    for i, s in enumerate(strategies):
        p = crn_patience(seed, i)
        agents.append(Agent(i, s, patience_limit=p))

    full_log = []
    decision_events = []
    traverse_events = []  # for congestion heatmap / road-share

    for t in range(T):
        # Decision + traversal steps
        for a in agents:
            if a.state == "WAITING" and a.current_node < nodes:
                a.decide(t, network[a.current_node], seed=seed, decision_events=(decision_events if record_decisions else None))

            elif a.state == "TRAVERSING":
                # track traversal (time x node x road)
                traverse_events.append({
                    "t": int(t),
                    "node": int(a.current_node),
                    "road": a.current_road.name if a.current_road else None,
                    "strategy": a.strategy
                })
                finished = a.step_traverse()
                if finished:
                    if a.current_node >= nodes:
                        a.state = "FINISHED"
                        a.arrival_time = int(t)
                    else:
                        a.state = "WAITING"

            if record_full_log:
                full_log.append({
                    "T": int(t),
                    "ID": int(a.id),
                    "Strat": a.strategy,
                    "Node": int(a.current_node),
                    "State": a.state,
                    "Regret": float(a.regret),
                    "Wait": int(a.wait_time),
                    "Drive": int(a.drive_time),
                    "Fuel": float(a.fuel_spent),
                    "Incidents": int(a.incidents_hit),
                })

    res_df = pd.DataFrame([{
        "ID": a.id,
        "Strat": a.strategy,
        "Arrival": a.arrival_time,
        "Wait": a.wait_time,
        "Drive": a.drive_time,
        "Regret": a.regret,
        "Fuel": a.fuel_spent,
        "Utility": -(a.wait_time + a.drive_time + 2.5 * a.fuel_spent),
        "Patience": a.patience_limit,
        "Incidents": a.incidents_hit,
    } for a in agents])

    log_df = pd.DataFrame(full_log) if record_full_log else pd.DataFrame()
    dec_df = pd.DataFrame(decision_events) if record_decisions else pd.DataFrame()
    traverse_df = pd.DataFrame(traverse_events) if traverse_events else pd.DataFrame()

    return res_df, log_df, dec_df, traverse_df


# =========================
# 4) METHOD A: Empirical ε–NE over strategy types
# =========================
def epsilon_NE_baseline(baseline_strategies, seeds=20, max_agents=None):
    """
    epsilon = max unilateral gain by switching strategy type
    Returns:
      epsilon, best_dev_tuple, top_devs_df
      best_dev_tuple = (agent_id, from_strat, to_strat, gain)
    """
    N = len(baseline_strategies)
    idxs = list(range(N)) if max_agents is None else list(range(min(max_agents, N)))

    # baseline expected utility per agent
    U_base = np.zeros(N, dtype=float)
    for s in range(seeds):
        res_df, _, _, _ = simulate(baseline_strategies, seed=s, T=DEFAULT_T, record_full_log=False, record_decisions=False)
        U_base += res_df.sort_values("ID")["Utility"].values
    U_base /= seeds

    best_gain = -1e18
    best_dev = None

    rows = []
    for i in idxs:
        cur = baseline_strategies[i]
        for alt in STRATS:
            if alt == cur:
                continue
            dev = baseline_strategies.copy()
            dev[i] = alt

            U_dev_i = 0.0
            for s in range(seeds):
                res_df, _, _, _ = simulate(dev, seed=s, T=DEFAULT_T, record_full_log=False, record_decisions=False)
                U_dev_i += float(res_df.loc[res_df["ID"] == i, "Utility"].iloc[0])
            U_dev_i /= seeds

            gain = U_dev_i - U_base[i]
            rows.append({"agent": i, "from": cur, "to": alt, "gain": gain})

            if gain > best_gain:
                best_gain = gain
                best_dev = (i, cur, alt, gain)

    top_devs_df = pd.DataFrame(rows).sort_values("gain", ascending=False)
    epsilon = max(0.0, best_gain)
    return epsilon, best_dev, top_devs_df


# =========================
# 5) METHOD B: Instantaneous road-choice local NE gap
# =========================
def _road_cost_given_flow_params(a_e, b_e, cap, flow):
    congestion = b_e * (flow if flow < cap else flow ** 1.8)
    return a_e + congestion

def _compute_local_NE_counts(num_choosers, roads, base_flows, max_iters=20000, seed=0):
    """
    Atomic congestion game NE via best-response dynamics among 'num_choosers' players.
    Costs depend on base_flows + x_r.
    Returns: dict road_name -> count
    """
    if num_choosers <= 0:
        return {r.name: 0 for r in roads}

    rng = np.random.default_rng(seed)
    R = len(roads)
    assign = rng.integers(0, R, size=num_choosers)
    x = np.bincount(assign, minlength=R).astype(int)

    a = [r.a_e for r in roads]
    b = [r.b_e for r in roads]
    cap = [r.capacity for r in roads]
    names = [r.name for r in roads]

    def cost_choose(r_idx, x_local):
        flow = base_flows[names[r_idx]] + x_local[r_idx]
        return _road_cost_given_flow_params(a[r_idx], b[r_idx], cap[r_idx], flow)

    for _ in range(max_iters):
        changed = False
        for j in range(num_choosers):
            cur = assign[j]
            x[cur] -= 1

            costs = []
            for r in range(R):
                x_try = x.copy()
                x_try[r] += 1
                costs.append(cost_choose(r, x_try))
            best = int(np.argmin(costs))

            assign[j] = best
            x[best] += 1
            if best != cur:
                changed = True

        if not changed:
            break

    return {names[r]: int(x[r]) for r in range(R)}

def instantaneous_NE_gap_time_series(strategies, seed=0, T=DEFAULT_T):
    """
    Returns gap_df: per decision batch (t,node):
      gap = L1(real_counts - ne_counts)/M
    """
    _, _, dec_df, _ = simulate(strategies, seed=seed, T=T, record_full_log=False, record_decisions=True)
    if dec_df.empty:
        return pd.DataFrame(columns=["t", "node", "M", "gap"])

    gaps = []
    for (t, node), grp in dec_df.groupby(["t", "node"]):
        avail_names = list(grp["avail"].iloc[0])
        _, network = make_network()
        roads_all = network[int(node)]
        roads = [r for r in roads_all if r.name in avail_names]

        base_flows = grp["base_flows"].iloc[0]
        M = len(grp)

        real_counts = grp["choice"].value_counts().to_dict()
        for r in avail_names:
            real_counts.setdefault(r, 0)

        ne_counts = _compute_local_NE_counts(
            num_choosers=M,
            roads=roads,
            base_flows=base_flows,
            seed=(seed * 100000 + int(t) * 101 + int(node))
        )

        l1 = sum(abs(real_counts[r] - ne_counts[r]) for r in avail_names)
        gap = l1 / max(1, M)
        gaps.append({"t": int(t), "node": int(node), "M": int(M), "gap": float(gap)})

    return pd.DataFrame(gaps)


# =========================
# 6) BETTER VISUALS
# =========================
def plot_baseline_summary(res_df, log_df,dec_df):
    # Plot 1: Utility by Strategy
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=res_df, x="Strat", y="Utility", palette="Set2")
    plt.title("1. Utility by Strategy")
    plt.tight_layout()
    plt.show()

    # Plot 2: Arrival Distribution
    plt.figure(figsize=(8, 6))
    sns.histplot(data=res_df, x="Arrival", hue="Strat", multiple="stack")
    plt.title("2. Arrival Distribution")
    plt.tight_layout()
    plt.show()

    if not log_df.empty:
        # Plot 3: Congestion Waves
        plt.figure(figsize=(8, 6))
        log_df[log_df["State"] == "TRAVERSING"].groupby(["T", "Strat"]).size().unstack().fillna(0).plot()
        plt.title("3. Congestion Waves (Traversal count)")
        plt.tight_layout()
        plt.show()

        # Plot 4: Myopic Regret Over Time
        if dec_df is not None and not dec_df.empty:
            plt.figure(figsize=(8, 6))
            sns.lineplot(
                data=dec_df,
                x="t",
                y="regret",
                hue="strategy",
                errorbar=("ci", 95)   # shows spread across agents’ decision regrets
            )
            plt.title("4. Myopic Regret at Decision Times (with CI across agents)")
            plt.tight_layout()
            plt.show()

def plot_NE_gap(gap_df, title_prefix="baseline", min_batch_M=None):
    if gap_df.empty:
        print("No decision batches found; NE gap plot skipped.")
        return

    g = gap_df.copy()
    if min_batch_M is not None:
        g = g[g["M"] >= min_batch_M]

    if g.empty:
        print(f"No batches with M >= {min_batch_M}.")
        return

    # weighted mean gap per time
    by_t = g.groupby("t").apply(lambda d: float((d["gap"] * d["M"]).sum() / d["M"].sum()))
    M_t = g.groupby("t")["M"].sum()

    plt.figure(figsize=(10, 4))
    plt.plot(by_t.index, by_t.values)
    plt.xlabel("time t")
    plt.ylabel("weighted gap to local NE")
    ttl = f"Weighted distance to instantaneous road-choice NE ({title_prefix})"
    if min_batch_M is not None:
        ttl += f" | only batches M≥{min_batch_M}"
    plt.title(ttl)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 3))
    plt.plot(M_t.index, M_t.values)
    plt.xlabel("time t")
    plt.ylabel("# choosers")
    plt.title(f"Decision mass over time ({title_prefix})")
    plt.tight_layout()
    plt.show()

def plot_road_choice_shares(dec_df, title="Road-choice shares over time"):
    if dec_df.empty:
        print("No decision events to plot road-choice shares.")
        return

    # road counts per time
    ct = dec_df.groupby(["t", "choice"]).size().unstack().fillna(0)
    # normalize to shares per time (avoid divide by zero)
    shares = ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

    plt.figure(figsize=(10, 4))
    plt.stackplot(shares.index, [shares[c].values for c in shares.columns], labels=list(shares.columns))
    plt.xlabel("time t")
    plt.ylabel("share of choices")
    plt.title(title)
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

def plot_congestion_heatmap(traverse_df, title="Congestion heatmap (traversal count)"):
    if traverse_df.empty:
        print("No traversal events to plot heatmap.")
        return

    # Count traversals by (t, node)
    piv = traverse_df.groupby(["t", "node"]).size().unstack().fillna(0)

    plt.figure(figsize=(12, 4))
    sns.heatmap(piv.T, cmap="viridis")
    plt.xlabel("time t")
    plt.ylabel("node")
    plt.title(title)
    plt.tight_layout()
    plt.show()

def print_top_devs(top_devs_df, k=10):
    print("\nTop unilateral deviations (estimated gains):")
    print(top_devs_df.head(k).to_string(index=False))


# =========================
# 7) OPTIONAL ANIMATION (same CRN model)
# =========================
def animate_run(strategies, seed=0, T=DEFAULT_T):
    nodes, network = make_network()
    agents = []
    colors = [plt.cm.tab10(i % 10) for i in range(len(strategies))]

    for i, s in enumerate(strategies):
        p = crn_patience(seed, i)
        a = Agent(i, s, patience_limit=p)
        a.color = colors[i]
        agents.append(a)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_xlim(-0.5, nodes + 0.5)
    ax.set_ylim(-0.2, 1.2)
    ax.set_yticks([0, 0.5, 1.0])
    ax.set_yticklabels(["Slow", "Std", "Exp"])
    ax.set_xlabel("Node index")
    ax.set_title("Dynamic Traffic Routing (CRN-consistent)")

    scat = ax.scatter([], [], c=[], s=[], edgecolors="k")

    def update(frame):
        for a in agents:
            if a.state == "WAITING" and a.current_node < nodes:
                a.decide(frame, network[a.current_node], seed=seed, decision_events=None)
            elif a.state == "TRAVERSING":
                finished = a.step_traverse()
                if finished:
                    if a.current_node >= nodes:
                        a.state = "FINISHED"
                        a.arrival_time = frame
                    else:
                        a.state = "WAITING"

        xs = [a.pos_x for a in agents]
        ys = [a.pos_y if a.state == "TRAVERSING" else 0.5 for a in agents]
        scat.set_offsets(np.c_[xs, ys])
        scat.set_color([a.color for a in agents])
        scat.set_sizes([100 if a.state == "WAITING" else 30 for a in agents])
        return scat,

    ani = FuncAnimation(fig, update, frames=T, interval=20, blit=True, repeat=False)
    plt.show()
    return ani


# =========================
# MAIN: run baseline + NE analysis + visuals
# =========================
if __name__ == "__main__":
    # ---- Baseline simulation (single seed for visuals) ----
    seed0 = 0
    res_df, log_df, dec_df, traverse_df = simulate(
        BASELINE_STRATEGIES, seed=seed0, T=DEFAULT_T,
        record_full_log=True, record_decisions=True
    )

    print("\n--- Baseline summary (seed=0) ---")
    print(res_df.groupby("Strat")[["Utility", "Wait", "Drive", "Fuel", "Incidents"]].mean().round(3))
    print("\n--- First 5 drivers (log excerpt) ---")
    if not log_df.empty:
        print(log_df[log_df["ID"] < 5].head(20).to_string(index=False))

    # Standard plots
    plot_baseline_summary(res_df, log_df, dec_df)

    # Better visuals
    plot_road_choice_shares(dec_df, title="Road-choice shares over time (decisions)")
    plot_congestion_heatmap(traverse_df, title="Congestion heatmap by node (traversal count)")

    # ---- Method A: epsilon-NE (fast default) ----
    eps, best_dev, top_devs = epsilon_NE_baseline(
        BASELINE_STRATEGIES,
        seeds=20,        # increase for tighter estimates
        max_agents=40    # set None for all 100 agents (slower)
    )

    print("\n=== Method A: Empirical ε–NE (strategy deviations) ===")
    print(f"epsilon ≈ {eps:.3f}")
    if best_dev is not None:
        i, frm, to, gain = best_dev
        print(f"Strongest deviation: agent {i} : {frm} -> {to}  |  gain ≈ {gain:.3f} utility units")
    print_top_devs(top_devs, k=10)

    # ---- Method B: local instantaneous NE gap ----
    gap_df = instantaneous_NE_gap_time_series(BASELINE_STRATEGIES, seed=seed0, T=DEFAULT_T)
    print("\n=== Method B: Instantaneous NE gap (first rows) ===")
    print(gap_df.head().to_string(index=False))

    # Weighted gap (recommended for report)
    plot_NE_gap(gap_df, title_prefix="baseline", min_batch_M=None)
    # Cleaner version focusing on meaningful batches
    plot_NE_gap(gap_df, title_prefix="baseline", min_batch_M=10)

    # ---- Optional: animate the SAME model ----
    # animate_run(BASELINE_STRATEGIES, seed=0, T=500)



--- Baseline summary (seed=0) ---
             Utility   Wait   Drive    Fuel  Incidents
Strat                                                 
Aggressive  -280.882   0.00  163.92  46.785       0.92
Risk-Averse -375.746   0.00  230.32  58.170       0.28
Social      -406.557   0.00  251.84  61.887       0.28
Strategic   -431.915  10.48  262.80  63.454       0.40

--- First 5 drivers (log excerpt) ---
 T  ID      Strat  Node      State  Regret  Wait  Drive  Fuel  Incidents
 0   0 Aggressive     0 TRAVERSING     0.0     0      0  8.75          0
 0   1 Aggressive     0 TRAVERSING     0.0     0      0  8.87          0
 0   2 Aggressive     0 TRAVERSING     0.0     0      0  8.99          0
 0   3 Aggressive     0 TRAVERSING     0.0     0      0  9.11          1
 0   4 Aggressive     0 TRAVERSING     0.0     0      0  9.23          0
 1   0 Aggressive     0 TRAVERSING     0.0     0      1  8.75          0
 1   1 Aggressive     0 TRAVERSING     0.0     0      1  8.87          0
 1   2 Aggre

TypeError: plot_baseline_summary() takes 2 positional arguments but 3 were given